# Post-Hoc Ensemble Evaluation & Deployment

This notebook allows you to perform a deep-dive evaluation of the best models found during your Genetic Algorithm runs and prepares them for actual use.

**Workflow:**
1. **Locate Results**: Find the latest experiment results in the `HFE_GA_experiments` folder.
2. **Reconstruct**: Re-hydrate the best ensemble architecture (models + feature masks) from the results CSV.
3. **Analyze**: Use `GA_results_explorer` to generate diagnostic plots of the search process.
4. **Validate**: Evaluate performance on the unseen **Test Set** and **Hold-out Validation Set**.
5. **Deploy**: Wrap the ensemble in a scikit-learn compatible class for production use.
6. **Serialize**: Save the fitted model to a portable file.
7. **Environment**: Define requirements for running the model on another server.

In [ ]:
import glob
import os
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from ml_grid.util.ensemble_classifier import SklearnEnsembleClassifier
from ml_grid.util.evaluate_ensemble_methods import EnsembleEvaluator
from ml_grid.util.GA_results_explorer import GA_results_explorer
from ml_grid.util.global_params import global_parameters

# 1. Setup paths
CONFIG_PATH = 'config.yml'
BASE_PROJECT_DIR = "HFE_GA_experiments"

# Load config to get metadata
global_params = global_parameters(config_path=CONFIG_PATH)
INPUT_DATA_PATH = global_params.input_csv_path
OUTCOME_VAR = f"outcome_var_{global_params.outcome_var_n}" # Dynamically load from config

### 2. Locate and Load Latest Results
We automatically target the most recently created experiment folder.

In [ ]:
list_of_runs = glob.glob(os.path.join(BASE_PROJECT_DIR, "*"))
if not list_of_runs:
    raise FileNotFoundError(f"No experiment runs found in {BASE_PROJECT_DIR}")

latest_run_dir = max(list_of_runs, key=os.path.getctime)
results_csv = os.path.join(latest_run_dir, "final_grid_score_log.csv")

if not os.path.exists(results_csv):
    # Fallback for different folder structures if necessary
    results_csv = os.path.join(latest_run_dir, "log_store_dataframe.csv")

print(f"Targeting Run: {latest_run_dir}")
results_df = pd.read_csv(results_csv)

# Initialize Evaluator
evaluator = EnsembleEvaluator(
    input_csv_path=INPUT_DATA_PATH,
    outcome_variable=OUTCOME_VAR,
    initial_param_dict={"resample": None},
    debug=False
)
print("Evaluator initialized with dataset splits.")

### 3. Comprehensive Run Analysis
Before focusing on the best model, we use the `GA_results_explorer` to visualize the entire evolution process. This generates plots for feature stability, convergence, and hyperparameter impact.

In [ ]:
explorer = GA_results_explorer(
    df=results_df,
    original_feature_names=evaluator.original_feature_names
)

# This generates and saves a full suite of plots to the run directory
explorer.run_all_plots(plot_dir=latest_run_dir)
print(f"All diagnostic plots generated and saved to: {latest_run_dir}")

# Display a few key insights inline
explorer.plot_algorithm_distribution_in_ensembles()

### 4. Final Performance Check
Check the performance on the Hold-out sets to ensure the GA didn't overfit.

In [ ]:
weighting_methods = ["unweighted", "de"]
test_res = evaluator.evaluate_on_test_set_from_df(results_df, weighting_methods)
val_res = evaluator.validate_on_holdout_set_from_df(results_df, weighting_methods)

print("\n--- PERFORMANCE SUMMARY ---")
if not test_res.empty and 'weighting_method' in test_res.columns:
    # Handle potential column name variations (auc vs auc_score)
    cols_to_show = [c for c in ['weighting_method', 'accuracy', 'auc', 'auc_score'] if c in test_res.columns]
    display(test_res[cols_to_show])

    # Create a comparison plot between GA validation performance and unseen test performance.
    comparison_data = []
    for _, row in test_res.iterrows():
        val_score = row.get('auc', row.get('auc_score'))
        # Ensure val_score is a number before appending
        if pd.notna(val_score):
            comparison_data.append({'Source': 'GA Validation (AUC)', 'Score': val_score, 'Method': row['weighting_method']})

        test_score = row.get('accuracy', np.nan) # Use np.nan for missing accuracy
        if pd.notna(test_score):
            comparison_data.append({'Source': 'Hold-out Test (Accuracy)', 'Score': test_score, 'Method': row['weighting_method']})

    comp_df = pd.DataFrame(comparison_data)
    plt.figure(figsize=(10, 5))
    sns.barplot(data=comp_df, x='Method', y='Score', hue='Source')
    plt.title("Generalization Gap: GA Validation (AUC) vs. Unseen Test (Accuracy)")
    plt.ylim(0, 1.1)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()
else:
    print("No evaluation results were returned. Check if 'best_ensemble' values in the CSV are valid strings.")

### 5. Reconstruct and Create Sklearn Classifier
This section demonstrates how to wrap the evolved ensemble into a standard scikit-learn compatible estimator.

In [ ]:
# Get best ensemble architecture from results
if results_df.empty:
    raise ValueError("The results log is empty. Cannot identify a best ensemble.")

best_col = 'auc' if 'auc' in results_df.columns else 'auc_score'
best_row = results_df.loc[results_df[best_col].idxmax()]
parsed_arch_list = evaluator._parse_ensemble(best_row['best_ensemble'])

if not parsed_arch_list:
    raise ValueError("Could not parse the ensemble architecture.")

my_ensemble = SklearnEnsembleClassifier(
    parsed_arch_list[0],
    evaluator.original_feature_names
)

my_ensemble.fit(evaluator.ml_grid_object.X_train, evaluator.ml_grid_object.y_train)
print("\nSklearn compatible ensemble is ready!")

### 6. Quick Prediction Example
Before analyzing the internal makeup, let's verify the ensemble can generate predictions on a few samples from the test set.

In [ ]:
# Select 5 rows for demonstration
if my_ensemble is not None:
    X_test = evaluator.ml_grid_object.X_test
    y_test = evaluator.ml_grid_object.y_test
    sample_x = X_test.head(5)
    sample_y = y_test.head(5).values

    preds = my_ensemble.predict(sample_x)
    probs = my_ensemble.predict_proba(sample_x)

    # Create a comparison table with 1D arrays to avoid dimensionality errors
    comparison = pd.DataFrame(data={
        'Actual': np.ravel(sample_y),
        'Predicted': np.ravel(preds),
        'Confidence': np.ravel(probs[:, 1])
    })
    display(comparison)
else:
    print("No ensemble classifier was created due to empty results. Skipping prediction example.")

### 7. Ensemble Makeup and Feature Insights
This section breaks down the internal structure of the evolved ensemble, showing model types and feature usage statistics.

In [ ]:
from collections import Counter

# 1. Model Type Distribution
model_types = [type(m[0]).__name__ for m in my_ensemble.fitted_models]
type_counts = Counter(model_types)

print("--- Base Learner Composition ---")
display(pd.Series(type_counts).to_frame('Count'))

plt.figure(figsize=(8, 4))
sns.barplot(x=list(type_counts.values()), y=list(type_counts.keys()))
plt.title("Distribution of Model Types in Ensemble")
plt.xlabel("Count")
plt.show()

# 2. Feature Count Statistics
feature_counts = [len(m[1]) for m in my_ensemble.fitted_models]
print("\n--- Feature Usage per Model ---")
print(f"- Total Base Learners: {len(my_ensemble.fitted_models)}")
print(f"- Avg Features per Model: {np.mean(feature_counts):.2f}")
print(f"- Min/Max Features: {min(feature_counts)} / {max(feature_counts)}")

# 3. Feature Frequency Analysis
all_used_features = []
for _, features, _ in my_ensemble.fitted_models:
    all_used_features.extend(features)

feat_freq = Counter(all_used_features)
most_common = feat_freq.most_common(10)

print("\n--- Top 10 Most Frequent Features ---")
top_feat_df = pd.DataFrame(most_common, columns=['Feature', 'Frequency'])
top_feat_df['Prevalence_%'] = (top_feat_df['Frequency'] / len(my_ensemble.fitted_models)) * 100
display(top_feat_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_feat_df, x='Prevalence_%', y='Feature', palette='viridis')
plt.title("Top 10 Most Influential Features (by Model Frequency)")
plt.show()

In [ ]:
# 4. Detailed Component Table
ensemble_summary = []
for i, (model, features, weight) in enumerate(my_ensemble.fitted_models):
    ensemble_summary.append({
        'Learner_ID': i + 1,
        'Model_Type': type(model).__name__,
        'Evolved_Weight': round(weight, 4),
        'Num_Features': len(features),
        'Feature_Sample': ", ".join(list(features)[:5]) + ("..." if len(features) > 5 else "")
    })

summary_df = pd.DataFrame(ensemble_summary).set_index('Learner_ID')
print("\n--- Detailed Ensemble Component View ---")
display(summary_df)

### 7. Deployment Readiness: Robustness & Latency
Before productionizing, we need to ensure the model handles data correctly even if column order changes, and measure its prediction speed.

In [ ]:
# 1. Feature Order Robustness Test
X_test = evaluator.ml_grid_object.X_test
X_shuffled = X_test.sample(frac=1, axis=1, random_state=42) # Shuffle column order

preds_orig = my_ensemble.predict(X_test.head(10))
preds_shuff = my_ensemble.predict(X_shuffled.head(10))

if np.array_equal(preds_orig, preds_shuff):
    print("✅ Column Order Robustness: PASSED (Model correctly maps features by name)")
else:
    print("❌ Column Order Robustness: FAILED")

# 2. Inference Latency Benchmarking
start_time = time.time()
n_samples = len(X_test)
_ = my_ensemble.predict(X_test)
end_time = time.time()

total_time = end_time - start_time
latency_per_sample = (total_time / n_samples) * 1000

print("\n--- Inference Performance ---")
print(f"- Total Rows Predicted: {n_samples}")
print(f"- Total Time: {total_time:.4f} seconds")
print(f"- Latency per Sample: {latency_per_sample:.4f} ms")

### 8. Model Persistence (Portability)
We serialize the fitted `my_ensemble` object using `joblib`. This file contains everything needed for inference.

In [ ]:
model_path = os.path.join(latest_run_dir, "deployed_ensemble_model.joblib")

# Save the model
joblib.dump(my_ensemble, model_path)
print(f"Model saved to: {model_path}")

# Verify by loading it back
loaded_model = joblib.load(model_path)

# Run a quick test prediction to ensure consistency
original_preds = my_ensemble.predict(evaluator.ml_grid_object.X_test.head(1))
loaded_preds = loaded_model.predict(evaluator.ml_grid_object.X_test.head(1))

if np.array_equal(original_preds, loaded_preds):
    print("✅ Serialization verified: Loaded model matches the original.")
else:
    print("❌ Serialization error: Prediction mismatch.")

### 9. Minimum Deployment Environment
To run this model on a different server, the target environment needs a subset of the project dependencies. 

**Requirements:**
1. **Python 3.10+**
2. **The `ensemble-genetic-algorithm` package**: The model relies on custom project classes. Since the package is not on PyPI yet, install it from the repository:
   ```bash
   ./setup.sh --cpu
   ```
3. **Standard Libraries**: `numpy`, `pandas`, `scikit-learn`, `joblib`, and `torch` (if using PyTorch base learners).

**Example Deployment Script:**

In [ ]:
deployment_code = """
import joblib
import pandas as pd

# 1. Load the model
model = joblib.load('deployed_ensemble_model.joblib')

# 2. Prepare new data (must be a DataFrame with original feature names)
new_data = pd.read_csv('new_unseen_data.csv')

# 3. Predict
predictions = model.predict(new_data)
probabilities = model.predict_proba(new_data)
"""
print("Minimum script required on target server:")
print(deployment_code)